In [1]:
import os
import numpy as np
import pandas as pd
from astropy.io import fits

# ============================================================
# SHARED SETUP
# ============================================================
tract_id = "3828"
bands = ['u', 'g', 'r', 'i', 'z', 'y']
tile_size = 80
image_size = 4100
patch_spacing = 4000  # confirmed empirically: patches are spaced 4000px apart

# --- FINAL verified patch selection (blendedness_truth, Duan et al. 2026 methodology) ---
patch_tiers = {
    "HIGH": ['0,6', '0,0', '1,0', '3,4', '0,2'],
    "MEAN": ['5,0', '5,1', '6,0', '4,1', '4,3'],
    "LOW":  ['2,4', '1,3', '3,1', '6,1', '4,6'],
}

def build_grid_positions(image_size, tile_size):
    positions = list(range(0, image_size - tile_size + 1, tile_size))
    if positions[-1] + tile_size < image_size:
        positions.append(image_size - tile_size)
    return positions

x_positions = build_grid_positions(image_size, tile_size)
y_positions = build_grid_positions(image_size, tile_size)

catalog = pd.read_parquet(r"processed\cleaned_catalog.parquet")


def process_patch(patch_id, quality_tier):
    raw_images_dir = os.path.join(r"RAW", r"IMAGES", quality_tier)
    tiles_out_dir = os.path.join(r"processed", r"TILES", quality_tier, patch_id)
    os.makedirs(tiles_out_dir, exist_ok=True)

    print(f"=== PATCH {patch_id} ({quality_tier}) ===")

    # FIXED: use patch_true, not patch_meas — must match the column used
    # for blendedness_truth stratification, or catalog rows won't correspond
    # to the patches actually selected.
    patch_cat = catalog[catalog['patch_true'] == patch_id].copy()
    print(f"Found {len(patch_cat)} total objects in patch {patch_id}.")

    x_raw = patch_cat['x'].values
    y_raw = patch_cat['y'].values

    # --- offset correction (confirmed needed) ---
    patch_col, patch_row = map(int, patch_id.split(','))
    x_offset = patch_col * patch_spacing
    y_offset = patch_row * patch_spacing
    x_coords = x_raw - x_offset
    y_coords = y_raw - y_offset

    if len(x_coords) > 0:
        print(f"x range (corrected): {x_coords.min():.1f} to {x_coords.max():.1f}")
        print(f"y range (corrected): {y_coords.min():.1f} to {y_coords.max():.1f}")

    print(f"Loading FITS images for patch {patch_id} ({quality_tier})...")
    image_data = {}
    for band in bands:
        file_name = f"calexp-{band}-{tract_id}-{patch_id}.fits"
        fits_path = os.path.join(raw_images_dir, band.upper(), file_name)
        with fits.open(fits_path) as hdul:
            image_data[band] = hdul[1].data

    saved_tiles = 0
    discarded_tiles = 0

    for y_min in y_positions:
        for x_min in x_positions:
            x_max = x_min + tile_size
            y_max = y_min + tile_size

            mask = (x_coords >= x_min) & (x_coords < x_max) & \
                   (y_coords >= y_min) & (y_coords < y_max)

            if np.any(mask):
                tile_bands = [image_data[band][y_min:y_max, x_min:x_max] for band in bands]
                stacked_tensor = np.stack(tile_bands, axis=0)
                stacked_tensor = np.nan_to_num(stacked_tensor, nan=0.0, posinf=0.0, neginf=0.0)
                np.save(os.path.join(tiles_out_dir, f"tile_x{x_min}_y{y_min}.npy"), stacked_tensor)
                saved_tiles += 1
            else:
                discarded_tiles += 1

    print(f"Saved: {saved_tiles} | Discarded: {discarded_tiles} | -> {tiles_out_dir}\n")
    return saved_tiles, discarded_tiles


# ============================================================
# RUN ALL 15 PATCHES
# ============================================================
results = {}
for tier, patch_list in patch_tiers.items():
    for patch_id in patch_list:
        saved, discarded = process_patch(patch_id, tier)
        results[(tier, patch_id)] = (saved, discarded)

print("="*60)
print("SUMMARY")
print("="*60)
total_saved = sum(s for s, d in results.values())
total_discarded = sum(d for s, d in results.values())
for (tier, patch_id), (saved, discarded) in results.items():
    print(f"{tier:5s} {patch_id:6s} saved={saved:5d} discarded={discarded:5d}")
print(f"\nTOTAL saved: {total_saved:,} | TOTAL discarded: {total_discarded:,}")

=== PATCH 0,6 (HIGH) ===
Found 2026 total objects in patch 0,6.
x range (corrected): 642.6 to 3995.4
y range (corrected): 0.4 to 3351.0
Loading FITS images for patch 0,6 (HIGH)...
Saved: 1140 | Discarded: 1564 | -> processed\TILES\HIGH\0,6

=== PATCH 0,0 (HIGH) ===
Found 2224 total objects in patch 0,0.
x range (corrected): 863.1 to 3999.3
y range (corrected): 559.0 to 3999.5
Loading FITS images for patch 0,0 (HIGH)...
Saved: 1213 | Discarded: 1491 | -> processed\TILES\HIGH\0,0

=== PATCH 1,0 (HIGH) ===
Found 2703 total objects in patch 1,0.
x range (corrected): 1.3 to 3998.3
y range (corrected): 590.4 to 3999.2
Loading FITS images for patch 1,0 (HIGH)...
Saved: 1480 | Discarded: 1224 | -> processed\TILES\HIGH\1,0

=== PATCH 3,4 (HIGH) ===
Found 2954 total objects in patch 3,4.
x range (corrected): -0.3 to 3999.0
y range (corrected): 2.5 to 3999.5
Loading FITS images for patch 3,4 (HIGH)...
Saved: 1684 | Discarded: 1020 | -> processed\TILES\HIGH\3,4

=== PATCH 0,2 (HIGH) ===
Found 2655

In [2]:
import pandas as pd
from pathlib import Path
import re

pattern = re.compile(r"^tile_x(\d+)_y(\d+)$")

def build_manifest(tiles_root):
    records = []
    for tier_dir in tiles_root.iterdir():
        if not tier_dir.is_dir():
            continue
        tier = tier_dir.name
        for patch_dir in tier_dir.iterdir():
            if not patch_dir.is_dir():
                continue
            patch_id = patch_dir.name
            for f in patch_dir.glob("*.npy"):
                m = pattern.match(f.stem)
                if not m:
                    print(f"Unrecognized filename, skipping: {f}")
                    continue
                x_pos, y_pos = int(m.group(1)), int(m.group(2))
                records.append({
                    "tile_path": str(f),
                    "tier": tier,
                    "patch_id": patch_id,
                    "x_pos": x_pos,
                    "y_pos": y_pos,
                })
    return pd.DataFrame(records)


# ============================================================
# MANIFEST 1 — TILES (80x80x6, channel-first)
# ============================================================
tiles_root = Path(r"processed/TILES")
manifest = build_manifest(tiles_root)

print("=== TILES manifest ===")
print(f"Total tiles: {len(manifest):,}")
print(manifest["tier"].value_counts())
print(manifest.groupby("patch_id").size().sort_values(ascending=False))

manifest.to_parquet(r"processed/tile_manifest.parquet", index=False)
print("Saved: processed/tile_manifest.parquet\n")

=== TILES manifest ===
Total tiles: 22,166
tier
LOW     7624
MEAN    7604
HIGH    6938
Name: count, dtype: int64
patch_id
2,4    1746
4,1    1698
4,3    1695
3,4    1684
1,3    1664
3,1    1650
5,1    1611
1,0    1480
5,0    1427
0,2    1421
4,6    1307
6,1    1257
0,0    1213
6,0    1173
0,6    1140
dtype: int64
Saved: processed/tile_manifest.parquet

